# Notebook 1 — Regresión Lineal Simple y Múltiple con `scikit-learn`

**Curso:** Fundamentos de Aprendizaje Automático
**Tema:** Supervised Learning , Regresión Lineal

## Objetivos de esta notebook

1. Entender cómo entrenar un modelo de **regresión lineal simple** (1 variable) con `sklearn`.
2. Entender cómo entrenar un modelo de **regresión lineal múltiple** (varias variables) con `sklearn`.
3. Interpretar los coeficientes (`w`), el intercepto (`b`) y las métricas de evaluación (MAE, MSE, R²).
4. Practicar con datasets nuevos descargados de Kaggle.
5. Implementar la **Ecuación Normal** (mínimos cuadrados vía álgebra matricial) y comparar el resultado con `sklearn`.

Trabajaremos primero con un dataset **100% numérico** (sin necesidad de transformación de columnas todavía) para enfocarnos en el modelo en sí.

## 0. Setup

Importamos las librerías que usaremos a lo largo de la notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Para que las gráficas se vean bien
plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['axes.grid'] = True

## 1. El dataset: `load_diabetes`

`sklearn` trae varios datasets "de juguete" (*toy datasets*) listos para practicar, sin necesidad de descargar nada. Usaremos `load_diabetes`, que contiene datos de 442 pacientes con diabetes.

- **Variables de entrada (X):** 10 variables fisiológicas ya numéricas (edad, sexo, índice de masa corporal, presión arterial, y 6 mediciones de sangre) — **ya vienen estandarizadas** (media 0).
- **Variable objetivo (y):** una medida cuantitativa de progresión de la enfermedad, un año después del diagnóstico.

Es un dataset ideal para empezar porque **no tiene columnas de texto**, nos enfocamos primero en el modelo, y dejamos la transformación de datos para el siguiente notebook.

In [ ]:
diabetes = load_diabetes(as_frame=True)

df = diabetes.frame  # DataFrame con features + target
print("Dimensiones:", df.shape)
df.head()

In [ ]:
df.describe()

In [ ]:
# Nombres de las columnas de features
diabetes.feature_names

## 2. Regresión Lineal Simple

Empezamos con el caso más sencillo: usar **una sola variable** (`bmi`, índice de masa corporal) para predecir `target`.

$$\hat{y} = w \cdot x + b$$

In [ ]:
# Seleccionamos una sola feature: bmi
X_simple = df[['bmi']]
y = df['target']

X_simple.shape, y.shape

### 2.1 Train / test split

Siempre separamos los datos en **entrenamiento** y **prueba** — el modelo aprende (`fit`) solo con train, y evaluamos qué tan bien generaliza con test (datos que nunca vio).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_simple, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)

### 2.2 Entrenar el modelo

`LinearRegression()` de sklearn ya implementa por debajo el algoritmo que aprendimos en clase (mínimos cuadrados). Solo llamamos `.fit(X, y)`.

In [ ]:
modelo_simple = LinearRegression()
modelo_simple.fit(X_train, y_train)

print("Coeficiente (w):", modelo_simple.coef_)
print("Intercepto (b):", modelo_simple.intercept_)

**Interpretación:** el coeficiente `w` nos dice cuánto cambia la predicción de `target` por cada unidad que aumenta `bmi`. El intercepto `b` es el valor predicho cuando `bmi = 0` (recuerda que aquí `bmi` ya está estandarizado, así que 0 representa el promedio de la población).

### 2.3 Predecir y visualizar

In [ ]:
y_pred_simple = modelo_simple.predict(X_test)

plt.scatter(X_test, y_test, color='steelblue', alpha=0.6, label='Datos reales')
plt.plot(X_test, y_pred_simple, color='darkorange', linewidth=2, label='Línea de regresión')
plt.xlabel('bmi (estandarizado)')
plt.ylabel('target (progresión de la enfermedad)')
plt.title('Regresión Lineal Simple: bmi → target')
plt.legend()
plt.show()

### 2.4 Evaluar el modelo

Usamos las métricas que vimos en clase: **MAE**, **MSE** y **R²**.

In [ ]:
mae = mean_absolute_error(y_test, y_pred_simple)
mse = mean_squared_error(y_test, y_pred_simple)
r2 = r2_score(y_test, y_pred_simple)

print(f"MAE : {mae:.2f}")
print(f"MSE : {mse:.2f}")
print(f"R²  : {r2:.3f}")

> **Pregunta para reflexionar:** con un solo feature (`bmi`), el R² probablemente no es muy alto. ¿Por qué? Piensa en cuántos factores influyen realmente en la progresión de una enfermedad — ¿es razonable esperar que una sola variable lo explique todo?

## 3. Regresión Lineal Múltiple

Ahora usamos **las 10 variables** del dataset para predecir `target`.

$$\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_{10} x_{10} + b$$

El código es prácticamente idéntico — `sklearn` maneja la multivariable de forma transparente.

In [ ]:
X_multi = df[diabetes.feature_names]  # las 10 columnas
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X_multi, y, test_size=0.2, random_state=42
)

modelo_multi = LinearRegression()
modelo_multi.fit(X_train, y_train)

print("Intercepto (b):", modelo_multi.intercept_)
print()
print("Coeficientes (w):")
for nombre, coef in zip(diabetes.feature_names, modelo_multi.coef_):
    print(f"  {nombre:8s}: {coef:8.2f}")

**Interpretación:** cada coeficiente representa el efecto de esa variable sobre `target`, **manteniendo las demás variables constantes**. Fíjate cuáles tienen los coeficientes con mayor magnitud (en valor absoluto) — esas son las variables que más "pesan" en la predicción.

In [ ]:
y_pred_multi = modelo_multi.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_multi)
mse = mean_squared_error(y_test, y_pred_multi)
r2 = r2_score(y_test, y_pred_multi)

print(f"MAE : {mae:.2f}")
print(f"MSE : {mse:.2f}")
print(f"R²  : {r2:.3f}")

### 3.1 Real vs. predicho

Cuando tenemos más de una variable, ya no podemos graficar una "línea" en 2D. Una forma común de visualizar qué tan bien predice el modelo es graficar **valores reales vs. valores predichos** — entre más cerca estén los puntos de la diagonal, mejor es el modelo.

In [ ]:
plt.scatter(y_test, y_pred_multi, color='seagreen', alpha=0.6)
lims = [min(y_test.min(), y_pred_multi.min()), max(y_test.max(), y_pred_multi.max())]
plt.plot(lims, lims, color='gray', linestyle='--', label='Predicción perfecta')
plt.xlabel('Valor real')
plt.ylabel('Valor predicho')
plt.title('Regresión Lineal Múltiple: Real vs. Predicho')
plt.legend()
plt.show()

### 3.2 Comparación simple vs. múltiple

| Modelo | Features usados | R² |
|---|---|---|
| Simple | 1 (`bmi`) | ver arriba |
| Múltiple | 10 (todas) | ver arriba |

**Pregunta para reflexionar:** ¿el R² mejoró al agregar más variables? ¿Tiene sentido que así sea? ¿Siempre agregar más variables mejora el modelo?

---
## 4. Ejercicios de práctica

Ahora te toca a ti. Vas a descargar dos datasets de Kaggle y replicar el flujo completo que acabamos de ver.

> **Nota:** si no tienes cuenta de Kaggle, puedes descargar el archivo `.csv` directamente desde la página del dataset (botón "Download") y subirlo manualmente a tu entorno de trabajo (Colab, Jupyter, etc.).

### Ejercicio 1 — Regresión Lineal Simple

**Dataset:** [Study Hours vs Student Scores](https://www.kaggle.com/datasets/douaabennoune/study-hours-student-scores-for-linear-regression)

Este dataset relaciona las horas de estudio de un estudiante con la calificación que obtuvo.

**Tareas:**
1. Descarga el dataset y cárgalo con `pd.read_csv(...)`.
2. Explora el dataset: `.head()`, `.info()`, `.describe()`. ¿Cuántas filas y columnas tiene? ¿Hay valores nulos?
3. Define `X` (horas de estudio) e `y` (calificación).
4. Haz el `train_test_split` (80% train / 20% test).
5. Entrena un `LinearRegression()`.
6. Imprime el coeficiente `w` y el intercepto `b`. **Interprétalos en una frase**: ¿cuánto sube la calificación por cada hora extra de estudio?
7. Grafica los puntos reales junto con la línea de regresión (igual que en la Sección 2.3).
8. Calcula MAE, MSE y R². ¿Este modelo predice mejor o peor que el de `bmi → target` que vimos en clase? ¿Por qué crees que pasa eso?

In [ ]:
# Ejercicio 1 - Tu código aquí

# 1. Cargar el dataset
# df_horas = pd.read_csv('nombre_del_archivo.csv')

# 2. Explorar
# df_horas.head()

# 3. Definir X e y


# 4. Train/test split


# 5. Entrenar el modelo


# 6. Coeficientes


# 7. Graficar


# 8. Métricas


### Ejercicio 2 — Regresión Lineal Múltiple

**Dataset:** [Advertising Dataset](https://www.kaggle.com/datasets/tawfikelmetwally/advertising-dataset)

Este dataset relaciona la inversión en publicidad en TV, radio y periódico con las ventas (`sales`) resultantes.

**Tareas:**
1. Descarga y carga el dataset.
2. Explora los datos. ¿Cuáles son las columnas de entrada (features) y cuál es la columna objetivo?
3. Define `X` (usa las 3 columnas de inversión publicitaria) e `y` (`sales`).
4. Haz el `train_test_split`.
5. Entrena un `LinearRegression()`.
6. Imprime los 3 coeficientes. **¿Cuál medio de publicidad (TV, radio, periódico) tiene el mayor impacto en las ventas según el modelo?**
7. Calcula MAE, MSE y R².
8. Haz la gráfica de "Real vs. Predicho" (como en la Sección 3.1).
9. **Extra:** entrena un segundo modelo usando solo 2 de las 3 variables (elimina la que menos aportó según el paso 6). ¿El R² cambió mucho? ¿Qué te dice esto sobre la importancia relativa de esa variable?

In [ ]:
# Ejercicio 2 - Tu código aquí

# 1. Cargar el dataset
# df_ads = pd.read_csv('nombre_del_archivo.csv')

# 2. Explorar


# 3. Definir X e y


# 4. Train/test split


# 5. Entrenar el modelo


# 6. Coeficientes


# 7. Métricas


# 8. Gráfica Real vs. Predicho


# 9. Extra: modelo con 2 variables


### Ejercicio 3 — La Ecuación Normal (mínimos cuadrados con álgebra matricial)

En clase vimos que la regresión lineal se puede resolver de dos formas:

1. **Gradiente descendente** (iterativo, lo usa `sklearn` internamente en muchos casos).
2. **Ecuación normal** (una fórmula cerrada, resuelve el problema en un solo paso):

$$w = (X^T X)^{-1} (X^T y)$$

En este ejercicio vas a implementar la ecuación normal **a mano usando NumPy**, y vas a comprobar que el resultado es (prácticamente) idéntico al que te da `sklearn`.

**Recuerda del material de clase:** la fórmula $w = (X^T X)^{-1}(X^T y)$, **por sí sola, no incluye el bias**. Para que sí lo incluya, hay que agregar una columna de puros 1's a la matriz `X` antes de hacer el cálculo — esa columna hace que el primer valor de `w` sea el intercepto `b`.

**Pasos sugeridos:**

1. Usa el dataset `diabetes` (regresión múltiple, las 10 variables) que ya usamos en la Sección 3.
2. Construye la matriz `X` como un array de NumPy (`X_train.values` o `X_train.to_numpy()`).
3. **Agrega una columna de 1's** al inicio de `X` — puedes usar `np.hstack()` o `np.column_stack()` junto con `np.ones((n_filas, 1))`.
4. Convierte `y_train` también a un array de NumPy.
5. Implementa la fórmula paso a paso usando funciones de NumPy:
   - Transponer: `X.T`
   - Multiplicar matrices: `X.T @ X` (el operador `@` hace multiplicación matricial)
   - Invertir una matriz: `np.linalg.inv(...)`
   - Junta todo para calcular `w = np.linalg.inv(X.T @ X) @ X.T @ y`
6. El primer valor del vector `w` resultante es el **bias**, y los siguientes 10 son los **coeficientes** de cada variable.
7. Compara estos valores con `modelo_multi.intercept_` y `modelo_multi.coef_` de la Sección 3. **¿Son iguales o muy parecidos?**

> **Tip:** si `np.linalg.inv()` te da un error o resultados extraños, revisa que no tengas columnas con multicolinealidad perfecta (dos columnas que sean combinación lineal exacta una de la otra) — en ese caso la matriz no es invertible.

In [ ]:
# Ejercicio 3 - Tu código aquí

# 1. Convertir X_train y y_train a arrays de numpy
# X_np = X_train.to_numpy()
# y_np = y_train.to_numpy()

# 2. Agregar columna de 1's (bias)
# unos = np.ones((X_np.shape[0], 1))
# X_bias = np.hstack([unos, X_np])

# 3. Aplicar la ecuación normal
# w_normal = np.linalg.inv(X_bias.T @ X_bias) @ X_bias.T @ y_np

# 4. Separar bias y coeficientes
# b_normal = w_normal[0]
# coefs_normal = w_normal[1:]

# 5. Comparar con sklearn
# print("Intercepto (ecuación normal):", b_normal)
# print("Intercepto (sklearn)        :", modelo_multi.intercept_)
# print()
# print("Coeficientes (ecuación normal):", coefs_normal)
# print("Coeficientes (sklearn)        :", modelo_multi.coef_)


**Pregunta final de reflexión:** si la ecuación normal da (casi) el mismo resultado que `sklearn` en un solo paso de álgebra matricial, ¿por qué en la práctica muchas veces se prefiere usar gradiente descendente en vez de la ecuación normal, especialmente con datasets muy grandes? (Pista: piensa en el costo computacional de invertir una matriz $X^T X$ cuando el número de variables es muy grande — esa operación es aproximadamente $O(d^3)$, donde $d$ es el número de columnas de `X`).